# NB5 · The Loop: agents, harnesses and workflows

**Agentic Systems Engineering · Lecture 5 · Vizuara · Dr. Rajat Dandekar**

Northbridge Cycles builds e-bikes in Manchester and runs on **Ledgerline ERP 7.2** (1998). Its assistant, **Ask
Northbridge**, has a brain (Lecture 2), knowledge (Lecture 3) and hands: 16 tools (Lecture 4). Today we open the loop
that uses the hands. Emma Clarke, a purchase executive, types on Monday 26 October 2026:

> **Order 2,000 BAT-500 packs from Voltway for the spring launch run.**

This notebook follows the lecture:

1. the smallest loop: fifteen lines, no limits
2. ReAct as text: Thought, Action, Observation, and the parser
3. the harness, one step at a time: Emma's order, the pause, the night as JSON, the morning
4. a workflow: the same order as written-down blocks; a run that waits for Daniel and resumes
5. the race: the workflow and the agent on the six jobs, scored from Ledgerline's tables

**No API key needed.** Every model call is replayed from our recorded runs (cassettes `L5v4_*`), so you see exactly
what happened in class. Five exercises are marked `TODO (exercise N)`, each with its solution behind
`SOLUTION = True`. Set it to `False` and try it yourself. The cells marked **A5** are where the assignment starts.

In [1]:
# Setup: install the Ledgerline sandbox and its recorded model replies. Runs in about a minute on Colab.
# REPLAY mode (the default when you have no key) plays back our recorded model calls, so every cell runs
# without an API key and shows the numbers from our own run. To call a model live, set a key below.
import os, sys, subprocess
COURSE_REPO = "https://github.com/RajatDandekar/agents-for-enterprises"   # course code (Ledgerline sandbox + reference agent)
try:
    import ledgerline  # already available (local checkout or Modal)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ledgerline-erp[all] @ git+{COURSE_REPO}.git#subdirectory=ledgerline"], check=True)
    import ledgerline
# os.environ["GEMINI_API_KEY"] = "..."        # optional: your own key, to run live
# os.environ["OPENROUTER_API_KEY"] = "..."    # optional: GPT, Claude, Llama ... via OpenRouter
os.environ.setdefault("LEDGER_LLM_MODE", "auto")  # auto = replay what was recorded, call live only for new requests
print("Ledgerline", ledgerline.__version__, "· model mode:", os.environ["LEDGER_LLM_MODE"])

Ledgerline 0.1.0 · model mode: replay


In [2]:
import json
import re

from ledgerline.llm import LLM
from ledgerline.daedalus import race as R
from ledgerline.daedalus.loop import SHIPPED, Agent
from ledgerline.daedalus.react import ReAct, parse_step
from ledgerline.daedalus.smallest import agent as smallest_agent
from ledgerline.daedalus.workflow import BLOCKS, SHIPPED_BLOCKS, Workflow

SOLUTION = True
ORDER = "Order 2,000 BAT-500 packs from Voltway for the spring launch run."


def cassette(name):
    """How this notebook reaches the model: replay from a recorded cassette (live only if you set a key)."""
    return lambda model: LLM(model, cassette=name, max_tokens=4096)


def pp(o, n=1200):
    s = json.dumps(o, indent=1, ensure_ascii=False, default=str)
    print(s if len(s) <= n else s[:n] + " …")


print("Six jobs:", ", ".join(j["id"] for j in R.JOBS))

Six jobs: spring, bells, nordic, closed, fallback, question


## 1 · The smallest loop

Think (call the model), check for words, act (run the tools it asked for), observe (put the results back), repeat.
Read the source: there is nothing that stops it except the model answering in words.

In [3]:
import inspect
from ledgerline.daedalus import smallest
print(inspect.getsource(smallest.agent))

def agent(question, toolbox, llm, system="You are Ask Northbridge."):
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": question}]
    while True:                                    # no step limit, no budget
        r = llm.chat(messages, tools=toolbox.schemas())         # 1. think
        messages.append(r.message())
        if not r.tool_calls:                               # 2. words: done
            return r.content
        for call in r.tool_calls:                                 # 3. act
            result = toolbox.call(call.name, call.arguments)
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(result, default=str)})  # 4. observe



In [4]:
tb = R.toolbox(R.new_erp())                       # a fresh Ledgerline, signed in as Emma, all 16 tools
llm = cassette("L5v4_nb_smallest")("gemini:gemini-3.8-flash")
print(smallest_agent("How many BAT-500 battery packs do we have in stock, and how many days is that?", tb, llm))
print(llm.totals())

We currently have **6,600 units** of **BAT-500** (Battery pack, 36 V 500 Wh) in stock, providing **11 days of cover**.

### Breakdown by Warehouse:
* **Manchester main stores (`WH-MAN-STR`):** 5,400 units
* **Leeds stores (`WH-LDS-STR`):** 1,200 units

*(None are currently allocated or on order; reorder level is 3,000 units.)*
{'calls': 2, 'tokens_in': 4834, 'tokens_out': 139, 'seconds': 5.28, 'cost_inr': 0.2494}


### TODO (exercise 1): give the smallest loop a step limit
Write `agent_with_limit(question, toolbox, llm, max_steps)`: the same loop, but after `max_steps` model calls it
returns `"(stopped: step limit)"` instead of going round again.

In [5]:
def agent_with_limit(question, toolbox, llm, max_steps=3):
    if not SOLUTION:
        raise NotImplementedError("exercise 1")
    messages = [{"role": "system", "content": "You are Ask Northbridge."}, {"role": "user", "content": question}]
    for _ in range(max_steps):
        r = llm.chat(messages, tools=toolbox.schemas())
        messages.append(r.message())
        if not r.tool_calls:
            return r.content
        for call in r.tool_calls:
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(toolbox.call(call.name, call.arguments), default=str)})
    return "(stopped: step limit)"


print(agent_with_limit("How many BAT-500 battery packs do we have in stock, and how many days is that?",
                       R.toolbox(R.new_erp()), cassette("L5v4_nb_smallest")("gemini:gemini-3.8-flash"), max_steps=1))

(stopped: step limit)


## 2 · ReAct as text

Before tool-calling APIs, the loop was a prompt: the tools are listed as text, the model writes `Thought:`,
`Action:` and `Action Input:`, and **our code** parses the action, runs it, and writes `Observation:` back.

In [6]:
react = ReAct(R.toolbox(R.new_erp()), LLM("gemini:gemini-3.8-flash", cassette="L5v4_react", max_tokens=2048))
print(react.respond("How many days of BAT-500 battery packs do we have in stock?"))
for s in react.steps:
    print(f"\n--- step {s['step']} · {s['tokens_in']:,} tokens read ---\n{s['raw'].strip()}")
    if s.get("observation") is not None:
        print("Observation:", json.dumps(s["observation"], default=str)[:300])

We currently have 11 days of cover for the BAT-500 battery packs across all warehouses. 
There are 6,600 units available in free stock (5,400 at Manchester main stores and 1,200 at Leeds stores), with none currently on order.

--- step 1 · 1,192 tokens read ---
Action: check_stock
Action Input: {"item_code":"BAT-500"}
Observation: {"ok": true, "item_code": "BAT-500", "description": "Battery pack, 36 V 500 Wh", "unit": "NOS", "warehouses": [{"warehouse": "WH-MAN-STR", "name": "Manchester main stores", "on_hand": 5400, "allocated": 0, "on_order": 0, "free": 5400}, {"warehouse": "WH-LDS-STR", "name": "Leeds stores", "on_hand": 1

--- step 2 · 1,414 tokens read ---
Thought: I can reply now.
Final Answer: We currently have 11 days of cover for the BAT-500 battery packs across all warehouses. 
There are 6,600 units available in free stock (5,400 at Manchester main stores and 1,200 at Leeds stores), with none currently on order.


### TODO (exercise 2): break the parser
`parse_step` turns one model reply into fields. Feed it three broken replies and print what it returns: a missing
`Action Input:` line, broken JSON, and a reply that writes its own `Observation:`.

In [7]:
broken = ["Thought: check stock\nAction: check_stock",
          'Thought: check stock\nAction: check_stock\nAction Input: {"item_code": "BAT-500"',
          'Action: check_stock\nAction Input: {"item_code": "BAT-500"}\nObservation: 9,999 packs']
if SOLUTION:
    for b in broken:
        st = parse_step(b)
        print({k: st[k] for k in ("action", "action_input", "error", "invented_observation")})

{'action': 'check_stock', 'action_input': None, 'error': "No 'Action Input:' line after the Action.", 'invented_observation': False}
{'action': 'check_stock', 'action_input': None, 'error': "Action Input is not valid JSON (Expecting ',' delimiter).", 'invented_observation': False}
{'action': 'check_stock', 'action_input': {'item_code': 'BAT-500'}, 'error': None, 'invented_observation': True}


## 3 · The harness, one step at a time

`Agent` is our harness around the loop: the prompt, the history, the tool list, the step limit, the token budget,
error recovery, the approval pause, and state across days. `say()` starts a message; each `step()` is one model call
plus the tools it asked for. This replays the run on the lecture's slides.

In [8]:
erp = R.new_erp()
emma = Agent(R.toolbox(erp), SHIPPED, llm_for=cassette("L5v4_story"))
emma.say(ORDER)
while not emma.done:
    s = emma.step()
    print(f"step {s['n']}: {s['tokens_in']:,} tokens read · words: {bool(s['text'].strip())} ·",
          [(c["name"], c["result"].get("ok")) for c in s["calls"]], "·", s["harness"][0])
print("\nReply:", emma.reply)

step 1: 2,198 tokens read · words: False · [('check_budget', True), ('supplier_certificates', True), ('create_purchase_request', True)] · step 1 of 12
step 2: 2,746 tokens read · words: True · [] · step 2 of 12

Reply: Here is the preview for the purchase request:

- Item: BAT-500 (2,000 packs)
- Supplier: Voltway Cells (VC-EIN-0042)
- Estimated value: £273,600
- Cost centre: PRD-07 (Spring launch run)
- Need-by date: 9 Nov 2026
- Approver: Daniel Okafor (DOKAFOR), as the value exceeds your £250,000 limit

Would you like me to submit this purchase request?


In [9]:
turn = emma.respond("Yes, go ahead.")
print("loop ended:", turn["end"], "· waiting:", emma.waiting)
print(turn["reply"])

loop ended: paused · waiting: {'doc': 'PR/2026/04417', 'approver': 'DOKAFOR'}
Purchase request PR/2026/04417 for 2,000 BAT-500 packs (£273,600) has been submitted.
Its status is WAITING FOR APPROVAL from Daniel Okafor (DOKAFOR).
Nothing is ordered yet; once Daniel approves it, the purchase order can be created and released to Voltway Cells.


### The night: state as JSON
Daniel may sign tomorrow. Save the agent as JSON, throw it away, and build a new one from the JSON in the morning.
Before the model is called, the harness reads Ledgerline: is it approved?

In [10]:
night = json.dumps(emma.state(), default=str)
print(f"{len(night):,} bytes of JSON; keys: {list(json.loads(night))}")
del emma
print("Daniel signs in his own inbox:", R.daniel_signs(erp))
morning = Agent.from_state(json.loads(night), R.toolbox(erp), llm_for=cassette("L5v4_story"))
t = morning.respond(R.MORNING)
for s in t["steps"]:
    print(s.get("note") or [(c["name"], c["result"].get("status")) for c in s["calls"]])
print("\nLedgerline now:", R.new_docs(erp)["orders"])

6,917 bytes of JSON; keys: ['user', 'config', 'messages', 'waiting', 'docs', 'failed']
Daniel signs in his own inbox: ['PR/2026/04417']
Read Ledgerline before the model: PR/2026/04417 is APPROVED by DOKAFOR. The job resumes.
[('create_purchase_order', 'APPROVED, NOT YET RELEASED')]
[('release_purchase_order', 'RELEASED to the supplier')]
[]

Ledgerline now: [{'po_no': 'PO/2026/01182', 'status': 'RL', 'sup_cd': 'VC-EIN-0042', 'cc_cd': 'PRD-07', 'pr_no': 'PR/2026/04417', 'approved_by': 'DOKAFOR', 'itm_cd': 'BAT-500', 'qty': 2000, 'rate': 119.0}]


### TODO (exercise 3): a stop rule of your own
Write `too_many_writes(turn)`: True if a message's loop called write tools more than twice. Run it over the three
turns above (`morning.turns` holds the morning; the day-1 turns are in the saved JSON's messages). Why might a
harness want this rule?

In [11]:
WRITES = {"create_purchase_request", "create_purchase_order", "release_purchase_order", "approve_purchase_request",
          "receive_goods"}


def too_many_writes(turn, limit=2):
    if not SOLUTION:
        raise NotImplementedError("exercise 3")
    n = sum(1 for s in turn["steps"] if s.get("kind") == "model" for c in s["calls"] if c["name"] in WRITES)
    return n > limit


print("morning:", too_many_writes(morning.turns[-1]))

morning: False


## 4 · A workflow: the steps written down

The same order as eleven blocks. One model call reads Emma's words into fields; every other step is a tool or a
gate in a fixed order. At "Wait for approval" a waiting run is saved as JSON and stops.

In [12]:
for b in SHIPPED_BLOCKS:
    t, kind, needs, gives, line = BLOCKS[b["block"]]
    print(f"{t:24s} {kind:9s} needs {needs} gives {gives}")

Read the request         model     needs ['text'] gives ['item_hint', 'quantity', 'supplier_hint', 'cost_centre', 'need_by']
Find the item            tool      needs ['item_hint'] gives ['item_code']
Check stock              tool      needs ['item_code'] gives ['free', 'cover_days']
Find the supplier        tool      needs ['supplier_hint'] gives ['supplier_code']
Check the certificate    gate      needs ['supplier_code', 'item_code'] gives ['certified']
Check the budget         gate      needs ['cost_centre'] gives ['budget_left']
Raise the request        tool      needs ['item_code', 'quantity', 'cost_centre', 'need_by'] gives ['request_no', 'request_status']
Wait for approval        approval  needs ['request_no'] gives ['approved_by']
Create the order         tool      needs ['supplier_code', 'request_no'] gives ['po_no']
Release the order        tool      needs ['po_no'] gives ['po_status']
Tell Emma                reply     needs [] gives []


In [13]:
erp = R.new_erp()
wf = Workflow(SHIPPED_BLOCKS)
run = wf.go(wf.start(R.BY_ID["spring"]["text"]), R.toolbox(erp), cassette("L5v4_race_workflow"))
for s in run["steps"]:
    print(f"{s['n']:2d} {s['title']:24s} {s['status']:8s} {s.get('note') or s.get('got') or ''}")
print("\nstatus:", run["status"], f"· saved as {run['saved_bytes']:,} bytes ·", run["reply"])

 1 Read the request         done     {'is_order': True, 'item_hint': 'BAT-500', 'quantity': 2000, 'supplier_hint': 'Voltway Cells', 'cost_centre': 'PRD-07', 'need_by': '2026-11-09'}
 2 Find the item            done     BAT-500
 3 Check stock              done     6,600 free, 11 days of cover
 4 Find the supplier        done     Voltway Cells (VC-EIN-0042)
 5 Check the certificate    done     passes: UN 38.3 valid
 6 Check the budget         done     open, £1,190,000 left
 7 Raise the request        done     PR/2026/04417 · WAITING FOR APPROVAL
 8 Wait for approval        waiting  PR/2026/04417 waits for DOKAFOR.

status: waiting · saved as 7,171 bytes · PR/2026/04417 is waiting for DOKAFOR to approve it. The run is saved; it carries on when Ledgerline says APPROVED.


In [14]:
print("Daniel signs:", R.daniel_signs(erp))
wf.resume(run, R.toolbox(erp), cassette("L5v4_race_workflow"))
print(run["status"], "·", run["reply"])

Daniel signs: ['PR/2026/04417']
done · Done: PO/2026/01182 for 2,000 BAT-500 is released to VC-EIN-0042 (approved by DOKAFOR).


### TODO (exercise 4): add a gate
Add a block `limit` ("Check the quantity", a gate) that stops the run when the quantity is above 5,000, and put it
after "Read the request". Run it on "The cheap supplier" job (2,000 packs: it passes) and see it in the steps.

In [15]:
if SOLUTION:
    from ledgerline.daedalus import workflow as W

    W.BLOCKS["limit"] = ("Check the quantity", "gate", ["quantity"], [], "Stop above 5,000 units.")

    def _b_limit(self, run, b, rec, tb, llm_for):
        if run["facts"]["quantity"] > 5000:
            raise W._Stop("stopped", "Over 5,000 units.", f"Stopped: {run['facts']['quantity']:,} is above 5,000 units.")
        rec["note"] = f"{run['facts']['quantity']:,} ≤ 5,000"

    W.Workflow._b_limit = _b_limit
    blocks = SHIPPED_BLOCKS[:1] + [{"block": "limit"}] + SHIPPED_BLOCKS[1:]
    r2 = W.Workflow(blocks).go(W.Workflow(blocks).start(R.BY_ID["nordic"]["text"]), R.toolbox(R.new_erp()),
                               cassette("L5v4_race_workflow"))
    for s in r2["steps"]:
        print(f"{s['n']:2d} {s['title']:24s} {s['status']:8s} {s.get('note') or ''}")

 1 Read the request         done     
 2 Check the quantity       done     2,000 ≤ 5,000
 3 Find the item            done     BAT-500
 4 Check stock              done     6,600 free, 11 days of cover
 5 Find the supplier        done     Nordic Cell Co. (NC-GDA-0063)
 6 Check the certificate    stopped  UN 38.3 expired for NC-GDA-0063.


## 5 · The race

The workflow and the agent on the six jobs, each in a fresh Ledgerline. Emma and Daniel are scripted the same way for
both; the score reads Ledgerline's tables. This replays our recorded race (the agent's first of three runs).

In [16]:
rows = []
for job in R.JOBS:
    w = R.run_workflow(job, SHIPPED_BLOCKS, cassette("L5v4_race_workflow"))
    a = R.run_agent(job, SHIPPED, cassette("L5v4_race_agent_r0"))
    rows += [w, a]
    print(f"{job['title']:28s} workflow {'right' if w['right'] else 'wrong':5s} {w['tokens_in']:>6,} tok · "
          f"agent {'right' if a['right'] else 'wrong':5s} {a['tokens_in']:>7,} tok")
print("\n", {who: R.totals([r for r in rows if r["who"] == who]) for who in ("workflow", "agent")})

The spring launch order      workflow right    208 tok · agent right  23,182 tok
Bells, within Emma's limit   workflow right    212 tok · agent right  25,842 tok


The cheap supplier           workflow right    209 tok · agent right  11,780 tok


The closed cost centre       workflow right    203 tok · agent right  10,816 tok


Cheapest, if allowed         workflow wrong    225 tok · agent right  35,138 tok


A question, not an order     workflow wrong    184 tok · agent right  11,739 tok

 {'workflow': {'right': 4, 'of': 6, 'model_calls': 6, 'tool_calls': 32, 'tokens_in': 1241, 'tokens_out': 363, 'usd': 0.00171, 'seconds': 0.0, 'failed': 0}, 'agent': {'right': 6, 'of': 6, 'model_calls': 37, 'tool_calls': 36, 'tokens_in': 118497, 'tokens_out': 3076, 'usd': 0.06848, 'seconds': 0.1, 'failed': 0}}


### TODO (exercise 5): score it your own way
Right answers are not everything. Write a score that also charges for tokens: 1 point per right job, minus 1 point
per 50,000 tokens read. Which side wins now? What would change your mind?

In [17]:
def my_score(rows, who):
    if not SOLUTION:
        raise NotImplementedError("exercise 5")
    rs = [r for r in rows if r["who"] == who]
    return round(sum(r["right"] for r in rs) - sum(r["tokens_in"] for r in rs) / 50_000, 2)


print({who: my_score(rows, who) for who in ("workflow", "agent")})

{'workflow': 3.98, 'agent': 3.63}


## A5 · Your job, both ways

Start here for the assignment (`A5.md`). Load your A4 tools with `ledgerline.course.l5.kit.FolderToolbox`, write your
workflow as a list of blocks (add your own `_b_<name>` methods, as in exercise 4), run your agent with
`Agent(your_toolbox, your_settings, llm_for=...)`, write six jobs of your own and a `score()` that reads your system's
tables, and race them with the loop above.

In [18]:
MY_JOBS = [
    # {"id": "plain1", "title": "...", "kind": "order", "text": "...", "answers": []},
]
print(len(MY_JOBS), "jobs of your own so far")

0 jobs of your own so far
